# 04 — Rhythm Supervision Targets (Colab + Drive)

Writes `features/rhythm/rhythm_song.csv` from **AcousticBrainz / Essentia** JSON (not a mel-proxy). GPU Off. Needs 00+01.

Downloads AcousticBrainz shards 00–02 by default (matching notebook 00) into `dataset/acousticbrainz/` if JSON files are not already on Drive.


## Colab + Drive (every notebook)

1. Open in **Google Colab**.
2. Run **Mount Drive** and click **Allow**.
3. Shared folder: `/content/drive/MyDrive/MTG_Instrument`
4. GPU **Off** for 00, 01, and 04. Branch training uses package scripts, not notebooks 02–09.
5. Do **not** re-download mels after notebook 00.


In [ ]:
!pip install -q tqdm


## Mount Drive


In [ ]:
from pathlib import Path
import os

DRIVE_ROOT = Path("/content/drive/MyDrive/MTG_Instrument")

if not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")
else:
    print("Drive already mounted")

for sub in ["dataset/logmel_songs", "annotations", "features", "checkpoints", "results/baselines", "results/proposed"]:
    (DRIVE_ROOT / sub).mkdir(parents=True, exist_ok=True)

os.environ["MTG_ROOT"] = str(DRIVE_ROOT)
print("Drive ready:", DRIVE_ROOT)


In [ ]:
from pathlib import Path
import os, json, random, re, shutil, socket, time, urllib.request
import numpy as np
import pandas as pd

DRIVE_ROOT = Path(os.environ.get("MTG_ROOT", "/content/drive/MyDrive/MTG_Instrument"))
ROOT = DRIVE_ROOT
MEL_DIR = ROOT / "dataset" / "logmel_songs"
MEL_CACHE = Path("/content/mel_cache")
MEL_CACHE.mkdir(parents=True, exist_ok=True)
ANN_DIR = ROOT / "annotations"
FEAT_DIR = ROOT / "features"
CKPT_DIR = ROOT / "checkpoints"
RESULTS_DIR = ROOT / "results"
BASELINE_RESULTS_DIR = RESULTS_DIR / "baselines"
BASELINE_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MANIFEST = ROOT / "dataset" / "song_manifest.csv"
RAW_ANN = "https://raw.githubusercontent.com/MTG/mtg-jamendo-dataset/master/data"
NEEDED_ANN = [
    "splits/split-0/autotagging_genre-train.tsv",
    "splits/split-0/autotagging_genre-validation.tsv",
    "splits/split-0/autotagging_genre-test.tsv",
    "splits/split-0/autotagging_instrument-train.tsv",
    "splits/split-0/autotagging_instrument-validation.tsv",
    "splits/split-0/autotagging_instrument-test.tsv",
]
SEED = 42
random.seed(SEED)
np.random.seed(SEED)


def check_internet(host="github.com", port=443, timeout=5) -> bool:
    try:
        socket.create_connection((host, port), timeout=timeout).close()
        return True
    except OSError:
        return False


def ensure_annotations():
    dest_train = ANN_DIR / "splits" / "split-0" / "autotagging_genre-train.tsv"
    if dest_train.exists():
        return
    if not check_internet():
        raise FileNotFoundError("Split TSVs missing and no Internet. Enable Internet and re-run.")
    print("Downloading annotation TSVs to Drive...")
    for rel in NEEDED_ANN:
        dest = ANN_DIR / rel
        dest.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(f"{RAW_ANN}/{rel}", dest)
        print(" ", dest)


def load_mel_npy(mel_abs, retries=5, pause=2.0):
    """Load mel from Drive with retries; cache on Colab disk to avoid FUSE drops."""
    mel_abs = Path(mel_abs)
    sid = normalize_track_id(mel_abs.stem) or mel_abs.stem.replace("/", "_")
    cached = MEL_CACHE / f"{sid}.npy"
    if cached.exists():
        try:
            return np.load(cached)
        except (OSError, ValueError):
            cached.unlink(missing_ok=True)

    last_err = None
    for attempt in range(retries):
        try:
            arr = np.load(mel_abs, mmap_mode=None)
            arr = np.asarray(arr, dtype=np.float32)
            np.save(cached, arr)
            return arr
        except (OSError, ValueError) as e:
            last_err = e
            if attempt + 1 < retries:
                time.sleep(pause * (attempt + 1))
    nbytes = mel_abs.stat().st_size if mel_abs.exists() else "missing"
    raise RuntimeError(
        f"Bad/truncated mel — re-download its shard in notebook 00: {mel_abs} "
        f"({nbytes} bytes on Drive). {last_err}"
    ) from last_err


def scan_bad_mels(df, label="manifest"):
    from tqdm.auto import tqdm

    bad = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"scan {label}"):
        try:
            load_mel_npy(row["mel_abs"])
        except Exception as e:
            bad.append({"song_id": str(row["song_id"]), "mel_abs": row["mel_abs"], "error": str(e)})
    if bad:
        out = RESULTS_DIR / f"bad_mels_{label}.json"
        out.write_text(json.dumps(bad, indent=2))
        print(f"WARNING: {len(bad)} bad mels → {out}")
    else:
        print(f"scan {label}: all {len(df)} mels OK (cache: {MEL_CACHE})")
    return bad


ensure_annotations()
print("ROOT   ", ROOT)
print("MEL_DIR", MEL_DIR, "npy=", len(list(MEL_DIR.rglob("*.npy"))))
print("ANN_DIR", ANN_DIR)
print("MANIFEST", MANIFEST, "exists=", MANIFEST.exists())


import csv

LOGMEL_SCHEMA_VERSION = "mtg_full_audio_logmel_windows_v1"
LOGMEL_N_MELS = 96
LOGMEL_WINDOW_FRAMES = 1366
LOGMEL_MAX_WINDOWS = 12
LOGMEL_SAMPLE_RATE = 12000
LOGMEL_HOP_LENGTH = 256


def normalize_track_id(raw) -> str | None:
    """Return the canonical seven-digit ID used by every project artifact."""
    value = str(raw).strip()
    match = re.fullmatch(r"(?:track_)?(\d+)", value, flags=re.IGNORECASE)
    if match is None:
        return None
    return f"{int(match.group(1)):07d}"


def iter_tsv_rows(path: Path):
    """Parse MTG's six fixed fields plus its variable number of tag columns."""
    with Path(path).open(newline="", encoding="utf-8", errors="strict") as handle:
        reader = csv.reader(handle, delimiter="\t")
        header = next(reader, None)
        expected = ["TRACK_ID", "ARTIST_ID", "ALBUM_ID", "PATH", "DURATION", "TAGS"]
        if header != expected:
            raise ValueError(f"unexpected MTG header in {path}: {header!r}")
        for line_number, fields in enumerate(reader, start=2):
            if not fields or all(not field.strip() for field in fields):
                continue
            if len(fields) < 6:
                raise ValueError(f"{path}:{line_number}: expected at least 6 tab-separated fields")
            song_id = normalize_track_id(fields[0])
            if song_id is None:
                raise ValueError(f"{path}:{line_number}: invalid TRACK_ID {fields[0]!r}")
            tags = tuple(field.strip() for field in fields[5:] if field.strip())
            yield {
                "TRACK_ID": fields[0],
                "song_id": song_id,
                "ARTIST_ID": fields[1],
                "ALBUM_ID": fields[2],
                "PATH": fields[3],
                "DURATION": fields[4],
                "TAGS": tags,
            }


def _canonical_logmel(raw, n_mels: int):
    mel = np.asarray(raw, dtype=np.float32)
    mel = np.squeeze(mel)
    if mel.ndim == 2:
        if mel.shape[0] == n_mels:
            full = mel
        elif mel.shape[1] == n_mels:
            full = mel.T
        else:
            raise ValueError(f"expected one log-Mel axis of size {n_mels}, got {mel.shape}")
    elif mel.ndim == 3:
        if mel.shape[1] == n_mels:
            full = mel.transpose(1, 0, 2).reshape(n_mels, -1)
        elif mel.shape[2] == n_mels:
            full = mel.transpose(2, 0, 1).reshape(n_mels, -1)
        else:
            raise ValueError(f"expected stacked log-Mels with {n_mels} bands, got {mel.shape}")
    else:
        raise ValueError(f"expected a 2D song log-Mel or 3D window stack, got {mel.shape}")

    if full.shape[1] == 0:
        raise ValueError("log-Mel has no time frames")
    return full


def _selected_chunk_indices(frame_count: int, n_frames: int, max_windows: int):
    if frame_count < 1 or n_frames < 1 or max_windows < 1:
        raise ValueError("frame counts and window limits must be positive")
    n_chunks = (frame_count + n_frames - 1) // n_frames
    if n_chunks <= max_windows:
        return np.arange(n_chunks, dtype=int)
    return np.linspace(0, n_chunks - 1, max_windows, dtype=int)


def logmel_window_plan(
    raw, *, n_mels: int, n_frames: int, max_windows: int,
    sample_rate: int = LOGMEL_SAMPLE_RATE, hop_length: int = LOGMEL_HOP_LENGTH,
):
    """Describe the exact ordered source-frame/time regions consumed by the model."""
    if sample_rate < 1 or hop_length < 1:
        raise ValueError("sample_rate and hop_length must be positive")
    full = _canonical_logmel(raw, n_mels)
    plan = []
    for output_index, chunk_index in enumerate(
        _selected_chunk_indices(full.shape[1], n_frames, max_windows)
    ):
        frame_start = int(chunk_index) * n_frames
        frame_end = min(frame_start + n_frames, full.shape[1])
        plan.append({
            "window_index": output_index,
            "chunk_index": int(chunk_index),
            "frame_start": frame_start,
            "frame_end_exclusive": frame_end,
            "valid_frames": frame_end - frame_start,
            "start_seconds": frame_start * hop_length / sample_rate,
            "end_seconds": frame_end * hop_length / sample_rate,
        })
    return plan


def segment_logmel_with_metadata(raw, *, n_mels: int, n_frames: int, max_windows: int):
    """Return windows plus the exact valid-frame counts and song-relative starts."""
    full = _canonical_logmel(raw, n_mels)
    plan = logmel_window_plan(
        full, n_mels=n_mels, n_frames=n_frames, max_windows=max_windows,
    )

    windows = np.zeros((max_windows, n_mels, n_frames), dtype=np.float32)
    mask = np.zeros(max_windows, dtype=np.float32)
    valid_frames = np.zeros(max_windows, dtype=np.int64)
    start_seconds = np.zeros(max_windows, dtype=np.float32)
    for item in plan:
        output_index = item["window_index"]
        start = item["frame_start"]
        piece = full[:, start:item["frame_end_exclusive"]]
        windows[output_index, :, :piece.shape[1]] = piece
        mask[output_index] = 1.0
        valid_frames[output_index] = item["valid_frames"]
        start_seconds[output_index] = item["start_seconds"]
    return windows, mask, valid_frames, start_seconds


def segment_logmel(raw, *, n_mels: int, n_frames: int, max_windows: int):
    """Return ordered, evenly covered song windows and a real-window mask."""
    windows, mask, _, _ = segment_logmel_with_metadata(
        raw, n_mels=n_mels, n_frames=n_frames, max_windows=max_windows,
    )
    return windows, mask


def split_annotation_path(split: str, subset: str = "genre") -> Path:
    if split not in {"train", "validation", "test"}:
        raise ValueError(f"unsupported split: {split!r}")
    path = ANN_DIR / "splits" / "split-0" / f"autotagging_{subset}-{split}.tsv"
    if not path.exists():
        raise FileNotFoundError(path)
    return path


def load_split_ids(split: str, subset: str = "genre") -> set[str]:
    path = split_annotation_path(split, subset)
    ids = {row["song_id"] for row in iter_tsv_rows(path)}
    if not ids:
        raise ValueError(f"no track IDs parsed from {path}")
    print(f"{split:12s} {len(ids):6d} ids ← {path}")
    return ids


def load_split_multihot(song_ids, subset: str, category: str):
    """Load fixed split-0 vocabulary and labels without treating missing rows as negatives."""
    annotations = {}
    split_for_song = {}
    vocabulary_by_split = {}
    prefix = f"{category}---"

    for split in ("train", "validation", "test"):
        path = split_annotation_path(split, subset)
        split_vocabulary = set()
        for row in iter_tsv_rows(path):
            sid = row["song_id"]
            if sid in split_for_song:
                raise ValueError(f"song {sid} occurs in both {split_for_song[sid]} and {split}")
            tags = {tag for tag in row["TAGS"] if tag.startswith(prefix)}
            unexpected = set(row["TAGS"]) - tags
            if unexpected:
                raise ValueError(f"unexpected non-{category} tags in {path}: {sorted(unexpected)[:3]}")
            if not tags:
                raise ValueError(f"song {sid} has no {category} tags in {path}")
            annotations[sid] = tags
            split_for_song[sid] = split
            split_vocabulary.update(tags)
        vocabulary_by_split[split] = split_vocabulary

    reference = vocabulary_by_split["train"]
    for split in ("validation", "test"):
        if vocabulary_by_split[split] != reference:
            missing = sorted(reference - vocabulary_by_split[split])
            extra = sorted(vocabulary_by_split[split] - reference)
            raise ValueError(
                f"{subset} vocabulary differs in {split}; missing={missing[:5]} extra={extra[:5]}"
            )

    label_names = sorted(reference)
    label_index = {name: index for index, name in enumerate(label_names)}
    normalized_ids = [normalize_track_id(sid) for sid in song_ids]
    if any(sid is None for sid in normalized_ids):
        raise ValueError("song_ids contains a non-MTG identifier")
    targets = np.zeros((len(normalized_ids), len(label_names)), dtype=np.float32)
    available = np.zeros(len(normalized_ids), dtype=bool)
    for row_index, sid in enumerate(normalized_ids):
        tags = annotations.get(sid)
        if tags is None:
            continue
        available[row_index] = True
        for tag in tags:
            targets[row_index, label_index[tag]] = 1.0
    return targets, label_names, available


import json
import os
import re
import hashlib
import time
from datetime import datetime
from pathlib import Path, PurePath


GPU_RUN_SCHEMA_VERSION = "gpu_run_request_v1"
GPU_JOBS = {
    "direct_cnn", "instrument_pretraining", "descriptor_fusion",
    "harmony_branch_screening",
}
GPU_STAGES = {"baseline", "pretraining", "branch_screening", "joint_training", "final_evaluation"}
JOB_STAGES = {
    "direct_cnn": {"baseline", "final_evaluation"},
    "instrument_pretraining": {"pretraining", "final_evaluation"},
    "descriptor_fusion": {"baseline", "final_evaluation"},
    "harmony_branch_screening": {"branch_screening"},
}
COHORT_SCHEMA_VERSION = "experiment_cohort_v1"
BUDGET_SCHEMA_VERSION = "gpu_budget_v1"


class GPUApprovalError(ValueError):
    pass


def _required_text(record, field):
    value = record.get(field)
    if not isinstance(value, str) or not value.strip():
        raise GPUApprovalError(f"{field} must be a non-empty string")
    return value.strip()


def _reject_placeholder(value, field):
    text = str(value).strip()
    if "REPLACE_" in text.upper():
        raise GPUApprovalError(f"{field} still contains a template placeholder")
    return text


def _safe_artifact_path(value, field):
    text = _required_text({field: value}, field)
    path = PurePath(text)
    if text in {"/", ".", "~"} or ".." in path.parts:
        raise GPUApprovalError(f"{field} is too broad or contains parent traversal")
    return text


def _load_cohort_artifact(path):
    try:
        cohort = json.loads(Path(path).read_text())
    except (OSError, json.JSONDecodeError) as error:
        raise GPUApprovalError(f"cannot read cohort artifact {path}: {error}") from error
    if not isinstance(cohort, dict) or cohort.get("schema_version") != COHORT_SCHEMA_VERSION:
        raise GPUApprovalError(f"cohort schema_version must be {COHORT_SCHEMA_VERSION!r}")
    source = cohort.get("source_manifest")
    if not isinstance(source, dict) or not re.fullmatch(r"[0-9a-f]{64}", str(source.get("sha256", ""))):
        raise GPUApprovalError("cohort source_manifest.sha256 must be a lowercase SHA-256")
    splits = cohort.get("splits")
    if not isinstance(splits, dict) or not splits:
        raise GPUApprovalError("cohort splits must be a non-empty object")
    unexpected = set(splits) - {"train", "validation", "test"}
    if unexpected:
        raise GPUApprovalError(f"cohort has unsupported splits: {sorted(unexpected)}")
    if not splits.get("train") or not splits.get("validation"):
        raise GPUApprovalError("cohort must contain non-empty train and validation splits")
    seen = set()
    normalized_splits = {}
    for split, song_ids in splits.items():
        if not isinstance(song_ids, list) or any(
            not isinstance(song_id, str) or not re.fullmatch(r"\d{7}", song_id)
            for song_id in song_ids
        ):
            raise GPUApprovalError(f"cohort {split} IDs must be seven-digit strings")
        if len(song_ids) != len(set(song_ids)):
            raise GPUApprovalError(f"cohort {split} contains duplicate song IDs")
        overlap = seen.intersection(song_ids)
        if overlap:
            raise GPUApprovalError(f"cohort song IDs occur in multiple splits: {sorted(overlap)[:3]}")
        seen.update(song_ids)
        normalized_splits[split] = list(song_ids)
    counts = cohort.get("counts")
    expected_counts = {split: len(ids) for split, ids in normalized_splits.items()}
    if counts != expected_counts:
        raise GPUApprovalError(f"cohort counts do not match IDs: expected {expected_counts}")
    canonical = json.dumps(normalized_splits, sort_keys=True, separators=(",", ":")).encode()
    expected_hash = hashlib.sha256(canonical).hexdigest()
    if cohort.get("cohort_sha256") != expected_hash:
        raise GPUApprovalError("cohort_sha256 does not match the frozen split IDs")
    return cohort


def _load_gpu_budget(path, record=None):
    try:
        ledger = json.loads(Path(path).read_text())
    except (OSError, json.JSONDecodeError) as error:
        raise GPUApprovalError(f"cannot read GPU budget ledger {path}: {error}") from error
    if not isinstance(ledger, dict) or ledger.get("schema_version") != BUDGET_SCHEMA_VERSION:
        raise GPUApprovalError(f"budget schema_version must be {BUDGET_SCHEMA_VERSION!r}")
    total = ledger.get("total_gpu_hours")
    if not isinstance(total, (int, float)) or isinstance(total, bool) or total <= 0:
        raise GPUApprovalError("budget total_gpu_hours must be positive")

    completed = ledger.get("completed_runs", [])
    reservations = ledger.get("reservations", [])
    if not isinstance(completed, list) or not isinstance(reservations, list):
        raise GPUApprovalError("budget completed_runs and reservations must be lists")
    completed_ids = set()
    completed_attempts = set()
    used = 0.0
    for item in completed:
        if not isinstance(item, dict):
            raise GPUApprovalError("each completed GPU run must be an object")
        run_id = _required_text(item, "run_id")
        experiment_id = _required_text(item, "experiment_id")
        attempt = item.get("attempt")
        actual = item.get("actual_gpu_hours")
        experiment_attempt = (experiment_id, attempt)
        if (
            run_id in completed_ids
            or attempt not in {1, 2}
            or experiment_attempt in completed_attempts
            or not isinstance(actual, (int, float))
            or isinstance(actual, bool)
            or actual < 0
        ):
            raise GPUApprovalError("completed GPU runs need unique IDs and non-negative actual hours")
        completed_ids.add(run_id)
        completed_attempts.add(experiment_attempt)
        used += float(actual)

    reservation_by_id = {}
    reserved_attempts = set()
    reserved = 0.0
    for item in reservations:
        if not isinstance(item, dict):
            raise GPUApprovalError("each GPU reservation must be an object")
        run_id = _required_text(item, "run_id")
        experiment_id = _required_text(item, "experiment_id")
        attempt = item.get("attempt")
        experiment_attempt = (experiment_id, attempt)
        estimate = item.get("estimated_gpu_hours")
        commit = _required_text(item, "code_commit")
        if (
            run_id in completed_ids
            or run_id in reservation_by_id
            or attempt not in {1, 2}
            or experiment_attempt in completed_attempts
            or experiment_attempt in reserved_attempts
        ):
            raise GPUApprovalError(
                "GPU run IDs and experiment attempts must be unique across the ledger"
            )
        if not isinstance(estimate, (int, float)) or isinstance(estimate, bool) or estimate <= 0:
            raise GPUApprovalError("GPU reservations require positive estimated_gpu_hours")
        if not re.fullmatch(r"[0-9a-fA-F]{7,40}", commit):
            raise GPUApprovalError("GPU reservation code_commit must be a Git hash")
        reservation_by_id[run_id] = item
        reserved_attempts.add(experiment_attempt)
        reserved += float(estimate)
    summary = {
        "total_gpu_hours": float(total),
        "used_gpu_hours": used,
        "reserved_gpu_hours": reserved,
        "unreserved_gpu_hours": float(total) - used - reserved,
        "overcommitted": used + reserved > float(total) + 1e-9,
    }
    if record is not None:
        reservation = reservation_by_id.get(record["run_id"])
        if reservation is None:
            raise GPUApprovalError("approved run has no matching GPU budget reservation")
        if (
            float(reservation["estimated_gpu_hours"]) != float(record["estimated_gpu_hours"])
            or reservation["code_commit"].lower() != record["code_commit"].lower()
            or reservation["experiment_id"] != record["experiment_id"]
            or reservation["attempt"] != record["attempt"]
        ):
            raise GPUApprovalError(
                "GPU budget reservation does not match run experiment, attempt, estimate, and commit"
            )
        available_before = summary["unreserved_gpu_hours"] + float(record["estimated_gpu_hours"])
        if abs(float(record["budget_remaining_before_gpu_hours"]) - available_before) > 1e-9:
            raise GPUApprovalError("run budget snapshot does not match the GPU budget ledger")
        summary["available_before_this_run_gpu_hours"] = available_before
    return ledger, summary


def _validate_required_gate_artifacts(record, artifact_paths):
    """Require semantic CPU evidence before the only registered harmony GPU screen."""
    if record["job"] != "harmony_branch_screening":
        return None
    if record["stage"] != "branch_screening":
        raise GPUApprovalError("harmony_branch_screening must use stage='branch_screening'")
    if record["experiment_id"] != "H1_temporal_chroma":
        raise GPUApprovalError(
            "only the registered H1_temporal_chroma experiment may use harmony GPU screening"
        )
    if record["seed"] != 42:
        raise GPUApprovalError("the first harmony GPU screen must use registered seed 42")

    decisions = []
    for value in artifact_paths:
        try:
            candidate = json.loads(Path(value).read_text())
        except (OSError, UnicodeDecodeError, json.JSONDecodeError):
            continue
        if (
            isinstance(candidate, dict)
            and candidate.get("schema_version") == "harmony_branch_screen_decision_v1"
        ):
            decisions.append(candidate)
    if len(decisions) != 1:
        raise GPUApprovalError(
            "harmony GPU screening requires exactly one harmony_branch_screen_decision_v1 artifact"
        )
    decision = decisions[0]
    if decision.get("decision") != "advance_to_gpu_registration":
        raise GPUApprovalError("harmony CPU branch gate did not advance to GPU registration")
    if decision.get("target_variant") != "temporal_chroma_v1":
        raise GPUApprovalError("harmony CPU gate does not approve temporal_chroma_v1")
    checks = decision.get("checks")
    if not isinstance(checks, dict) or not checks or not all(value is True for value in checks.values()):
        raise GPUApprovalError("harmony CPU branch gate does not contain all passing checks")
    for field in ("cohort_sha256", "screen_dataset_sha256", "source_policy_sha256", "source_report_sha256"):
        if not re.fullmatch(r"[0-9a-f]{64}", str(decision.get(field, ""))):
            raise GPUApprovalError(f"harmony CPU branch gate has invalid {field}")
    return decision


def _validate_repeat_evidence(record, evidence_path):
    """Allow one repeat only for a registered ambiguous comparison or recorded failure."""
    if record["attempt"] != 2:
        return None
    try:
        evidence = json.loads(Path(evidence_path).read_text())
    except (OSError, UnicodeDecodeError, json.JSONDecodeError) as error:
        raise GPUApprovalError(f"repeat evidence is not valid JSON: {error}") from error
    if not isinstance(evidence, dict):
        raise GPUApprovalError("repeat evidence must be a JSON object")

    if evidence.get("schema_version") == "paired_bootstrap_comparison_v1":
        if evidence.get("decision") != "ambiguous":
            raise GPUApprovalError("paired repeat evidence must have decision='ambiguous'")
        expected = {
            "comparison_id": record["comparison_id"],
            "control_run_id": record["control_run_id"],
            "candidate_run_id": record["repeat_of"],
            "metric": record["validation"]["metric"],
            "min_effect": record["validation"]["min_effect"],
            "confidence": record["validation"]["confidence"],
            "seed": record["validation"]["bootstrap_seed"],
            "control_sha256": record["control_artifact_sha256"],
        }
        for field, value in expected.items():
            if evidence.get(field) != value:
                raise GPUApprovalError(f"repeat evidence {field} does not match the run request")
        if not re.fullmatch(r"[0-9a-f]{64}", str(evidence.get("candidate_sha256", ""))):
            raise GPUApprovalError("repeat evidence candidate_sha256 is invalid")
        return evidence

    if evidence.get("schema_version") == "gpu_run_termination_v1":
        prior = evidence.get("gpu_run")
        if evidence.get("status") != "terminated_without_result" or not isinstance(prior, dict):
            raise GPUApprovalError("termination repeat evidence is not a failed run record")
        if prior.get("run_id") != record["repeat_of"]:
            raise GPUApprovalError("termination evidence does not describe repeat_of")
        reason = evidence.get("reason")
        if reason not in {
            "no_complete_validation_epoch",
            "embedding_export_cap_reached",
            "runtime_exception",
        }:
            raise GPUApprovalError("termination evidence has an unsupported failure reason")
        return evidence

    raise GPUApprovalError(
        "repeat evidence must be an ambiguous paired comparison or GPU termination ledger"
    )


def validate_gpu_run_record(record, *, expected_job=None, require_approved=True):
    if not isinstance(record, dict):
        raise GPUApprovalError("run record must be a JSON object")
    if record.get("schema_version") != GPU_RUN_SCHEMA_VERSION:
        raise GPUApprovalError(f"schema_version must be {GPU_RUN_SCHEMA_VERSION!r}")

    run_id = _required_text(record, "run_id")
    if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_.-]{2,63}", run_id):
        raise GPUApprovalError("run_id must be 3-64 safe filename characters")
    experiment_id = _required_text(record, "experiment_id")
    if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_.-]{2,63}", experiment_id):
        raise GPUApprovalError("experiment_id must be 3-64 safe filename characters")
    attempt = record.get("attempt")
    if not isinstance(attempt, int) or isinstance(attempt, bool) or attempt not in {1, 2}:
        raise GPUApprovalError("attempt must be 1 or 2; broader repeat sweeps are forbidden")
    repeat_of = record.get("repeat_of")
    if attempt == 1 and repeat_of not in {None, ""}:
        raise GPUApprovalError("attempt 1 must not declare repeat_of")
    if attempt == 2:
        repeat_of = _required_text(record, "repeat_of")
        if repeat_of == run_id:
            raise GPUApprovalError("repeat_of must differ from run_id")
        _required_text(record, "repeat_reason")
        repeat_artifact = _safe_artifact_path(
            record.get("repeat_evidence_artifact"), "repeat_evidence_artifact"
        )
        repeat_hash = _required_text(record, "repeat_evidence_sha256")
        if record.get("status") == "approved" and not re.fullmatch(r"[0-9a-f]{64}", repeat_hash):
            raise GPUApprovalError("repeat_evidence_sha256 must be a lowercase SHA-256")
    job = _required_text(record, "job")
    if job not in GPU_JOBS:
        raise GPUApprovalError(f"unsupported job: {job!r}")
    if expected_job is not None and job != expected_job:
        raise GPUApprovalError(f"record job {job!r} does not approve {expected_job!r}")
    stage = _required_text(record, "stage")
    if stage not in GPU_STAGES:
        raise GPUApprovalError(f"unsupported stage: {stage!r}")
    if stage not in JOB_STAGES[job]:
        raise GPUApprovalError(f"job {job!r} cannot run at stage {stage!r}")

    status = _required_text(record, "status")
    if status not in {"planned", "approved"}:
        raise GPUApprovalError("status must be 'planned' or 'approved'")
    if require_approved and status != "approved":
        raise GPUApprovalError("GPU execution requires status='approved'")
    _required_text(record, "question")
    _required_text(record, "single_change")
    code_commit = _required_text(record, "code_commit")
    cohort_artifact = _safe_artifact_path(record.get("cohort_artifact"), "cohort_artifact")
    artifact_dir = _safe_artifact_path(record.get("artifact_dir"), "artifact_dir")

    checks = record.get("cheaper_checks")
    if not isinstance(checks, list) or not checks:
        raise GPUApprovalError("cheaper_checks must contain at least one passed CPU/preflight check")
    for index, check in enumerate(checks):
        if not isinstance(check, dict):
            raise GPUApprovalError(f"cheaper_checks[{index}] must be an object")
        _required_text(check, "name")
        _required_text(check, "evidence")
        if check.get("status") != "passed":
            raise GPUApprovalError(f"cheaper_checks[{index}] is not passed")
        evidence_artifact = _safe_artifact_path(
            check.get("artifact"), f"cheaper_checks[{index}].artifact"
        )
        evidence_hash = _required_text(check, "sha256")
        if status == "approved" and not re.fullmatch(r"[0-9a-f]{64}", evidence_hash):
            raise GPUApprovalError(
                f"cheaper_checks[{index}].sha256 must be a lowercase SHA-256"
            )

    reused = record.get("reused_artifacts")
    if not isinstance(reused, list):
        raise GPUApprovalError("reused_artifacts must be a list (possibly empty)")
    normalized_reused = []
    for index, item in enumerate(reused):
        if not isinstance(item, dict):
            raise GPUApprovalError(f"reused_artifacts[{index}] must be an object")
        reused_path = _safe_artifact_path(
            item.get("path"), f"reused_artifacts[{index}].path"
        )
        reused_hash = _required_text(item, "sha256")
        if status == "approved" and not re.fullmatch(r"[0-9a-f]{64}", reused_hash):
            raise GPUApprovalError(
                f"reused_artifacts[{index}].sha256 must be a lowercase SHA-256"
            )
        normalized_reused.append({"path": reused_path, "sha256": reused_hash})

    max_epochs = record.get("max_epochs")
    if not isinstance(max_epochs, int) or isinstance(max_epochs, bool) or not 1 <= max_epochs <= 100:
        raise GPUApprovalError("max_epochs must be an integer in [1, 100]")
    max_wall_minutes = record.get("max_wall_minutes")
    if not isinstance(max_wall_minutes, (int, float)) or isinstance(max_wall_minutes, bool) or max_wall_minutes <= 0:
        raise GPUApprovalError("max_wall_minutes must be positive")
    if max_wall_minutes > 120 and not str(record.get("wall_cap_override_reason", "")).strip():
        raise GPUApprovalError("runs above 120 minutes require wall_cap_override_reason")
    estimated_gpu_hours = record.get("estimated_gpu_hours")
    if not isinstance(estimated_gpu_hours, (int, float)) or isinstance(estimated_gpu_hours, bool):
        raise GPUApprovalError("estimated_gpu_hours must be numeric")
    if estimated_gpu_hours <= 0 or estimated_gpu_hours > max_wall_minutes / 60:
        raise GPUApprovalError("estimated_gpu_hours must be positive and no larger than the wall cap")
    if status == "approved":
        remaining_gpu_hours = record.get("budget_remaining_before_gpu_hours")
        if (
            not isinstance(remaining_gpu_hours, (int, float))
            or isinstance(remaining_gpu_hours, bool)
            or remaining_gpu_hours < estimated_gpu_hours
        ):
            raise GPUApprovalError(
                "budget_remaining_before_gpu_hours must cover estimated_gpu_hours"
            )
        _safe_artifact_path(record.get("budget_ledger"), "budget_ledger")
    seed = record.get("seed")
    if not isinstance(seed, int) or isinstance(seed, bool):
        raise GPUApprovalError("seed must be one integer")

    validation = record.get("validation")
    if not isinstance(validation, dict):
        raise GPUApprovalError("validation must be an object")
    if validation.get("split") != "validation":
        raise GPUApprovalError("model selection must use the validation split")
    _required_text(validation, "metric")
    min_effect = validation.get("min_effect")
    if not isinstance(min_effect, (int, float)) or isinstance(min_effect, bool) or min_effect < 0:
        raise GPUApprovalError("validation.min_effect must be non-negative")
    comparison_stage = stage in {"branch_screening", "joint_training"}
    if comparison_stage and status == "approved" and min_effect <= 0:
        raise GPUApprovalError(
            "approved branch/joint comparisons require a positive validation.min_effect"
        )
    confidence = validation.get("confidence")
    if not isinstance(confidence, (int, float)) or isinstance(confidence, bool) or not 0.5 < confidence < 1:
        raise GPUApprovalError("validation.confidence must be between 0.5 and 1")
    if not isinstance(validation.get("bootstrap_seed"), int) or isinstance(validation.get("bootstrap_seed"), bool):
        raise GPUApprovalError("validation.bootstrap_seed must be one integer")

    evaluate_test = record.get("evaluate_test")
    if not isinstance(evaluate_test, bool):
        raise GPUApprovalError("evaluate_test must be boolean")
    if evaluate_test and stage != "final_evaluation":
        raise GPUApprovalError("test evaluation is allowed only at final_evaluation")
    if evaluate_test:
        _required_text(record, "test_authorization_reason")

    if status == "approved":
        if not re.fullmatch(r"[0-9a-fA-F]{7,40}", code_commit):
            raise GPUApprovalError("approved code_commit must be a 7-40 character Git hash")
        _required_text(record, "approved_by")
        approved_at = _required_text(record, "approved_at")
        try:
            parsed_approval_time = datetime.fromisoformat(approved_at.replace("Z", "+00:00"))
        except ValueError as error:
            raise GPUApprovalError("approved_at must be an ISO-8601 timestamp") from error
        if parsed_approval_time.tzinfo is None:
            raise GPUApprovalError("approved_at must include a timezone")
        for field in ("question", "single_change", "cohort_artifact", "artifact_dir"):
            _reject_placeholder(record[field], field)
        for index, check in enumerate(checks):
            _reject_placeholder(check["evidence"], f"cheaper_checks[{index}].evidence")
            _reject_placeholder(check["artifact"], f"cheaper_checks[{index}].artifact")
        for index, item in enumerate(normalized_reused):
            _reject_placeholder(item["path"], f"reused_artifacts[{index}].path")
        if attempt == 2:
            for field, value in (
                ("repeat_of", repeat_of),
                ("repeat_reason", record["repeat_reason"]),
                ("repeat_evidence_artifact", repeat_artifact),
            ):
                _reject_placeholder(value, field)

    if comparison_stage:
        comparison_id = _required_text(record, "comparison_id")
        control_run_id = _required_text(record, "control_run_id")
        if control_run_id == run_id:
            raise GPUApprovalError("control_run_id must differ from run_id")
        control_artifact = _safe_artifact_path(
            record.get("control_artifact"), "control_artifact"
        )
        control_hash = _required_text(record, "control_artifact_sha256")
        if status == "approved" and not re.fullmatch(r"[0-9a-f]{64}", control_hash):
            raise GPUApprovalError("control_artifact_sha256 must be a lowercase SHA-256")
        if status == "approved":
            for field, value in (
                ("comparison_id", comparison_id),
                ("control_run_id", control_run_id),
                ("control_artifact", control_artifact),
            ):
                _reject_placeholder(value, field)

    normalized = dict(record)
    normalized["cohort_artifact"] = cohort_artifact
    normalized["artifact_dir"] = artifact_dir
    normalized["reused_artifacts"] = normalized_reused
    if attempt == 2:
        normalized["repeat_evidence_artifact"] = repeat_artifact
    if comparison_stage:
        normalized["control_artifact"] = control_artifact
    return normalized


def require_gpu_run_approval(device, expected_job):
    if str(device).split(":", 1)[0] != "cuda":
        print("CPU execution: GPU approval record is not required.")
        return None
    record_path = os.environ.get("GPU_RUN_RECORD", "").strip()
    if not record_path:
        raise GPUApprovalError(
            "CUDA training is locked. Set GPU_RUN_RECORD to an approved JSON record after CPU gates pass."
        )
    path = Path(record_path)
    if not path.is_file():
        raise GPUApprovalError(f"GPU_RUN_RECORD does not exist: {path}")
    record = validate_gpu_run_record(json.loads(path.read_text()), expected_job=expected_job)
    cohort = Path(record["cohort_artifact"])
    if not cohort.is_absolute():
        cohort = path.parent / cohort
    if not cohort.is_file():
        raise GPUApprovalError(f"approved cohort artifact does not exist: {cohort}")
    cohort_data = _load_cohort_artifact(cohort)
    if record["evaluate_test"] and not cohort_data["splits"].get("test"):
        raise GPUApprovalError("final test evaluation requires a non-empty frozen test cohort")
    if record.get("control_artifact"):
        control = Path(record["control_artifact"])
        if not control.is_absolute():
            control = path.parent / control
        if not control.is_file():
            raise GPUApprovalError(f"approved control artifact does not exist: {control}")
        digest = hashlib.sha256()
        with control.open("rb") as handle:
            for chunk in iter(lambda: handle.read(1024 * 1024), b""):
                digest.update(chunk)
        if digest.hexdigest() != record["control_artifact_sha256"]:
            raise GPUApprovalError("approved control artifact SHA-256 does not match")
        record["resolved_control_artifact"] = str(control.resolve())
    resolved_check_artifacts = []
    for index, check in enumerate(record["cheaper_checks"]):
        evidence = Path(check["artifact"])
        if not evidence.is_absolute():
            evidence = path.parent / evidence
        if not evidence.is_file():
            raise GPUApprovalError(
                f"approved cheaper-check artifact does not exist: {evidence}"
            )
        digest = hashlib.sha256()
        with evidence.open("rb") as handle:
            for chunk in iter(lambda: handle.read(1024 * 1024), b""):
                digest.update(chunk)
        if digest.hexdigest() != check["sha256"]:
            raise GPUApprovalError(
                f"approved cheaper-check artifact SHA-256 does not match at index {index}"
            )
        resolved_check_artifacts.append(str(evidence.resolve()))
    record["resolved_cheaper_check_artifacts"] = resolved_check_artifacts
    gate_decision = _validate_required_gate_artifacts(record, resolved_check_artifacts)
    if gate_decision is not None:
        record["harmony_cpu_gate"] = gate_decision
    resolved_reused_artifacts = []
    for index, item in enumerate(record["reused_artifacts"]):
        reused_path = Path(item["path"])
        if not reused_path.is_absolute():
            reused_path = path.parent / reused_path
        if not reused_path.is_file():
            raise GPUApprovalError(f"approved reused artifact does not exist: {reused_path}")
        digest = hashlib.sha256()
        with reused_path.open("rb") as handle:
            for chunk in iter(lambda: handle.read(1024 * 1024), b""):
                digest.update(chunk)
        if digest.hexdigest() != item["sha256"]:
            raise GPUApprovalError(
                f"approved reused artifact SHA-256 does not match at index {index}"
            )
        resolved_reused_artifacts.append(str(reused_path.resolve()))
    record["resolved_reused_artifacts"] = resolved_reused_artifacts
    if record["attempt"] == 2:
        repeat_evidence = Path(record["repeat_evidence_artifact"])
        if not repeat_evidence.is_absolute():
            repeat_evidence = path.parent / repeat_evidence
        if not repeat_evidence.is_file():
            raise GPUApprovalError(f"approved repeat evidence does not exist: {repeat_evidence}")
        digest = hashlib.sha256()
        with repeat_evidence.open("rb") as handle:
            for chunk in iter(lambda: handle.read(1024 * 1024), b""):
                digest.update(chunk)
        if digest.hexdigest() != record["repeat_evidence_sha256"]:
            raise GPUApprovalError("approved repeat evidence SHA-256 does not match")
        _validate_repeat_evidence(record, repeat_evidence)
        record["resolved_repeat_evidence"] = str(repeat_evidence.resolve())
    budget_path = Path(record["budget_ledger"])
    if not budget_path.is_absolute():
        budget_path = path.parent / budget_path
    if not budget_path.is_file():
        raise GPUApprovalError(f"GPU budget ledger does not exist: {budget_path}")
    _, budget_summary = _load_gpu_budget(budget_path, record)
    requested_test = os.environ.get("EVALUATE_TEST", "0") == "1"
    if requested_test != record["evaluate_test"]:
        raise GPUApprovalError("EVALUATE_TEST must exactly match the approved run record")
    record["record_path"] = str(path.resolve())
    record["resolved_cohort_artifact"] = str(cohort.resolve())
    record["cohort_sha256"] = cohort_data["cohort_sha256"]
    record["cohort_counts"] = cohort_data["counts"]
    record["resolved_budget_ledger"] = str(budget_path.resolve())
    record["budget_summary"] = budget_summary
    print(f"GPU run approved: {record['run_id']} by {record['approved_by']}")
    return record


def apply_approved_cohort(manifest, record, manifest_path):
    """Restrict a manifest to the exact approved IDs before creating data loaders."""
    if record is None:
        return manifest
    manifest_path = Path(manifest_path)
    if not manifest_path.is_file():
        raise GPUApprovalError(f"manifest does not exist: {manifest_path}")
    if "song_id" not in manifest.columns or "split" not in manifest.columns:
        raise GPUApprovalError("manifest must contain song_id and split columns")
    if manifest["song_id"].duplicated().any():
        raise GPUApprovalError("manifest contains duplicate song IDs")
    cohort = _load_cohort_artifact(record["resolved_cohort_artifact"])
    digest = hashlib.sha256()
    with manifest_path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    if digest.hexdigest() != cohort["source_manifest"]["sha256"]:
        raise GPUApprovalError("current manifest SHA-256 does not match the frozen cohort source")
    approved_split = {
        song_id: split
        for split, song_ids in cohort["splits"].items()
        for song_id in song_ids
    }
    manifest_split = dict(zip(manifest["song_id"].astype(str), manifest["split"].astype(str)))
    missing = sorted(set(approved_split) - set(manifest_split))
    if missing:
        raise GPUApprovalError(f"approved cohort has {len(missing)} IDs absent from manifest: {missing[:3]}")
    wrong = sorted(
        song_id for song_id, split in approved_split.items()
        if manifest_split[song_id] != split
    )
    if wrong:
        raise GPUApprovalError(f"approved cohort split mismatch for IDs: {wrong[:3]}")
    selected = manifest[manifest["song_id"].astype(str).isin(approved_split)].copy()
    if len(selected) != len(approved_split):
        raise GPUApprovalError("manifest filtering did not produce the exact approved cohort")
    print("Applied approved cohort:", cohort["counts"], cohort["cohort_sha256"])
    return selected.reset_index(drop=True)


def approved_run_limits(record, *, default_epochs, requested_wall_minutes):
    if not isinstance(default_epochs, int) or isinstance(default_epochs, bool) or default_epochs < 1:
        raise GPUApprovalError("default_epochs must be a positive integer")
    if not isinstance(requested_wall_minutes, (int, float)) or isinstance(requested_wall_minutes, bool):
        raise GPUApprovalError("requested_wall_minutes must be numeric")
    if requested_wall_minutes <= 0:
        raise GPUApprovalError("requested_wall_minutes must be positive")
    if record is None:
        return int(default_epochs), float(requested_wall_minutes)
    return (
        min(int(default_epochs), int(record["max_epochs"])),
        min(float(requested_wall_minutes), float(record["max_wall_minutes"])),
    )


def write_gpu_termination_ledger(
    path, *, device, started_at, record, reason, max_epochs, max_wall_minutes
):
    """Persist consumed time before deliberately aborting a capped run."""
    elapsed = time.perf_counter() - started_at
    payload = {
        "schema_version": "gpu_run_termination_v1",
        "status": "terminated_without_result",
        "reason": reason,
        "device": str(device),
        "max_epochs": int(max_epochs),
        "max_wall_minutes": float(max_wall_minutes),
        "gpu_run": record,
        "wall_seconds": elapsed,
        "gpu_wall_hours": elapsed / 3600 if str(device).split(":", 1)[0] == "cuda" else 0.0,
    }
    destination = Path(path)
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(json.dumps(payload, indent=2))
    print("termination runtime", payload)
    return payload


## Download AcousticBrainz JSON (if missing) + extract rhythm fields


In [ ]:
import subprocess
from tqdm.auto import tqdm

if not MANIFEST.exists():
    raise FileNotFoundError("Run 01 first")
manifest = pd.read_csv(MANIFEST)
manifest["song_id"] = manifest["song_id"].astype(str).map(lambda s: normalize_track_id(s) or s)

AB_DIR = ROOT / "dataset" / "acousticbrainz"
AB_DIR.mkdir(parents=True, exist_ok=True)
AB_SHARDS = [0, 1, 2]  # must match the default mel subset from notebook 00
AB_URL = "https://cdn.freesound.org/mtg-jamendo/raw_30s/acousticbrainz"


def download_ab_shards():
    n_json = len(list(AB_DIR.rglob("*.json")))
    if n_json > 0:
        print(f"AcousticBrainz already on Drive: {n_json} JSON under {AB_DIR}")
        return
    if not check_internet("cdn.freesound.org") and not check_internet():
        raise RuntimeError(
            "No AcousticBrainz JSON on Drive and no Internet. "
            "Enable Internet, or run the official MTG script:\n"
            "  python3 scripts/download/download.py --dataset raw_30s "
            "--type acousticbrainz --from mtg-fast --unpack --remove "
            f"{AB_DIR}"
        )
    print("Downloading AcousticBrainz shards 00–02 to", AB_DIR)
    for i in AB_SHARDS:
        marker = AB_DIR / f".ab_shard_{i:02d}_done"
        if marker.exists():
            print(f"AB shard {i:02d} already done — skip")
            continue
        tar_name = f"raw_30s_acousticbrainz-{i:02d}.tar.gz"
        tar_path = AB_DIR / tar_name
        url = f"{AB_URL}/{tar_name}"
        print("Downloading", url)
        subprocess.check_call(["wget", "-q", "-O", str(tar_path), url])
        subprocess.check_call(["tar", "-xzf", str(tar_path), "-C", str(AB_DIR)])
        tar_path.unlink(missing_ok=True)
        marker.write_text("ok")
        print(f"AB shard {i:02d} saved")


def index_ab_json(root: Path) -> dict[str, Path]:
    idx = {}
    for p in root.rglob("*.json"):
        sid = normalize_track_id(p.stem)
        if sid:
            idx[sid] = p
    return idx


def _scalar(v):
    """Unwrap AcousticBrainz scalar or {mean: ...} stats. Never invent a default."""
    if v is None:
        return None
    if isinstance(v, dict):
        if "mean" in v:
            return _scalar(v["mean"])
        return None
    if isinstance(v, (list, tuple)):
        return None
    try:
        x = float(v)
    except (TypeError, ValueError):
        return None
    if np.isnan(x):
        return None
    return x


def rhythm_from_ab(doc: dict) -> dict | None:
    block = doc.get("rhythm")
    if not isinstance(block, dict):
        return None
    bpm = _scalar(block.get("bpm"))
    if bpm is None:
        return None
    beats_pos = block.get("beats_position")
    if not isinstance(beats_pos, (list, tuple)):
        beats_pos = []
    beats_count = _scalar(block.get("beats_count"))
    if beats_count is None and beats_pos:
        beats_count = float(len(beats_pos))
    intervals = np.diff(np.asarray(beats_pos, dtype=np.float64)) if len(beats_pos) > 1 else None
    feat = {
        "bpm": bpm,
        "beats_count": beats_count,
        "beats_loudness_mean": _scalar(block.get("beats_loudness")),
        "bpm_histogram_first_peak_bpm": _scalar(block.get("bpm_histogram_first_peak_bpm")),
        "bpm_histogram_first_peak_spread": _scalar(block.get("bpm_histogram_first_peak_spread")),
        "bpm_histogram_first_peak_weight": _scalar(block.get("bpm_histogram_first_peak_weight")),
        "onset_rate": _scalar(block.get("onset_rate")),
        "danceability": _scalar(block.get("danceability")),
        "beat_interval_mean": float(np.mean(intervals)) if intervals is not None else None,
        "beat_interval_std": float(np.std(intervals)) if intervals is not None else None,
    }
    return feat


download_ab_shards()
ab_index = index_ab_json(AB_DIR)
print("AcousticBrainz JSON indexed:", len(ab_index))

rows, missing = [], []
for _, rec in tqdm(manifest.iterrows(), total=len(manifest), desc="rhythm"):
    sid = str(rec["song_id"])
    path = ab_index.get(sid)
    if path is None:
        missing.append({"song_id": sid, "reason": "no_acousticbrainz_json"})
        continue
    try:
        doc = json.loads(path.read_text(encoding="utf-8", errors="replace"))
        feat = rhythm_from_ab(doc)
        if feat is None:
            missing.append({"song_id": sid, "reason": "json_missing_rhythm.bpm", "path": str(path)})
            continue
        feat.update({"song_id": sid, "source": "acousticbrainz", "split": rec["split"]})
        rows.append(feat)
    except Exception as e:
        missing.append({"song_id": sid, "reason": str(e), "path": str(path)})

n_mels = int(len(manifest))
n_ab_disk = int(len(ab_index))
n_written = int(len(rows))
n_excluded = int(len(missing))
print(f"songs with mels (manifest):     {n_mels}")
print(f"AcousticBrainz JSON on disk:    {n_ab_disk}")
print(f"overlap written to CSV:         {n_written}")
print(f"excluded (no/invalid AB JSON):  {n_excluded}")
if n_written == 0:
    raise RuntimeError("No overlapping AcousticBrainz rhythm rows — download AB shards and re-run.")

df = pd.DataFrame(rows)
out = FEAT_DIR / "rhythm"
out.mkdir(parents=True, exist_ok=True)
csv_path = out / "rhythm_song.csv"
df.to_csv(csv_path, index=False)
(RESULTS_DIR / "04_missing_acousticbrainz.json").write_text(json.dumps(missing, indent=2))
summary = {
    "n_manifest_mels": n_mels,
    "n_acousticbrainz_json": n_ab_disk,
    "n_overlap_written": n_written,
    "n_excluded": n_excluded,
    "csv": str(csv_path),
    "source": "acousticbrainz",
}
(RESULTS_DIR / "04_rhythm_summary.json").write_text(json.dumps(summary, indent=2))
print("wrote", csv_path, "rows=", n_written)
print(json.dumps(summary, indent=2))
df.head()
